In [1]:
# Pregunta: ¿qué conductores tienen mayor actividad una vez que relacionamos sus métricas con el registro maestro?
# La unión añade nombres legibles a las medidas calculadas a partir de los turnos.

import csv
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)

with (ROOT / "data" / "timesheet.csv").open(encoding="utf-8", newline="") as file:
    timesheet = list(csv.DictReader(file))

with (ROOT / "data" / "drivers.csv").open(encoding="utf-8", newline="") as file:
    drivers = list(csv.DictReader(file))

len(timesheet), len(drivers)

(1768, 34)

In [2]:
# Estos operadores ya se estudiaron en P519; aquí se usan para resolver el caso.

def map_pairs(records, mapper):
    pairs = []

    for record in records:
        pairs.extend(mapper(record))

    return pairs


def group_by_key(pairs):
    grouped_values = {}

    for key, value in pairs:
        grouped_values.setdefault(key, []).append(value)

    return grouped_values


def reduce_by_key(grouped_values, reducer):
    return [reducer(key, values) for key, values in grouped_values.items()]


def inner_join_by_key(left_pairs, right_pairs, joiner):
    right_values = dict(right_pairs)

    return [
        joiner(key, left_value, right_values[key])
        for key, left_value in left_pairs
        if key in right_values
    ]

In [3]:
# Esta regla del caso produce las parejas de métricas parciales a partir de los turnos.

def map_timesheet_record(record):
    return [
        (
            record["driverId"],
            (
                float(record["hours-logged"]),
                float(record["miles-logged"]),
                1,
            ),
        )
    ]


mapped_pairs = map_pairs(timesheet, map_timesheet_record)

mapped_pairs[:5]

[('10', (70.0, 3300.0, 1)),
 ('10', (70.0, 3300.0, 1)),
 ('10', (60.0, 2800.0, 1)),
 ('10', (70.0, 3100.0, 1)),
 ('10', (70.0, 3200.0, 1))]

In [4]:
# Esta regla reduce las semanas de un conductor a las métricas que se unirán al maestro.

def summarize_driver_metrics(driver_id, values):
    total_hours = sum(hours for hours, _, _ in values)
    total_miles = sum(miles for _, miles, _ in values)
    weeks = sum(week_count for _, _, week_count in values)

    return driver_id, {
        "total_hours": round(total_hours, 2),
        "total_miles": round(total_miles, 2),
        "weeks": weeks,
    }

In [5]:
# La reducción conserva driverId como clave para relacionar el resultado con otra fuente.

driver_metric_pairs = reduce_by_key(
    group_by_key(mapped_pairs),
    summarize_driver_metrics,
)
driver_metric_pairs.sort(key=lambda pair: int(pair[0]))

driver_metric_pairs[:5]

[('10', {'total_hours': 3232.0, 'total_miles': 147150.0, 'weeks': 52}),
 ('11', {'total_hours': 3642.0, 'total_miles': 179300.0, 'weeks': 52}),
 ('12', {'total_hours': 2639.0, 'total_miles': 135962.0, 'weeks': 52}),
 ('13', {'total_hours': 2727.0, 'total_miles': 134126.0, 'weeks': 52}),
 ('14', {'total_hours': 2781.0, 'total_miles': 136624.0, 'weeks': 52})]

In [6]:
# Esta regla del caso transforma el maestro de conductores en parejas de la misma clave.

def map_driver_record(record):
    return [(record["driverId"], {"name": record["name"]})]


driver_name_pairs = map_pairs(drivers, map_driver_record)

driver_name_pairs[:5]

[('10', {'name': 'George Vetticaden'}),
 ('11', {'name': 'Jamie Engesser'}),
 ('12', {'name': 'Paul Coddin'}),
 ('13', {'name': 'Joe Niemiec'}),
 ('14', {'name': 'Adis Cesir'})]

In [7]:
# Esta regla decide cómo combinar las dos fuentes para responder la pregunta de la operación.

def combine_driver_activity(driver_id, metrics, driver):
    return {"driverId": driver_id, **driver, **metrics}

In [8]:
# Esta consulta SQL expresa el resultado de la unión; los operadores producen el mismo resultado.
#
# SELECT
#     drivers.name,
#     metrics.total_hours,
#     metrics.total_miles,
#     metrics.weeks
# FROM drivers
# INNER JOIN metrics
#     ON drivers.driverId = metrics.driverId
# ORDER BY metrics.total_miles DESC;

driver_activity = inner_join_by_key(
    driver_metric_pairs,
    driver_name_pairs,
    combine_driver_activity,
)
driver_activity.sort(key=lambda record: record["total_miles"], reverse=True)

with (ROOT / "submission" / "driver_activity.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=list(driver_activity[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(driver_activity)

driver_activity[:10]

[{'driverId': '11',
  'name': 'Jamie Engesser',
  'total_hours': 3642.0,
  'total_miles': 179300.0,
  'weeks': 52},
 {'driverId': '10',
  'name': 'George Vetticaden',
  'total_hours': 3232.0,
  'total_miles': 147150.0,
  'weeks': 52},
 {'driverId': '33',
  'name': 'Sridhara Sabbella',
  'total_hours': 2759.0,
  'total_miles': 139285.0,
  'weeks': 52},
 {'driverId': '25',
  'name': 'Jean-Philippe Playe',
  'total_hours': 2723.0,
  'total_miles': 139180.0,
  'weeks': 52},
 {'driverId': '39',
  'name': 'David Kaiser',
  'total_hours': 2745.0,
  'total_miles': 138788.0,
  'weeks': 52},
 {'driverId': '15',
  'name': 'Rohit Bakshi',
  'total_hours': 2734.0,
  'total_miles': 138750.0,
  'weeks': 52},
 {'driverId': '35',
  'name': 'Emil Siemes',
  'total_hours': 2728.0,
  'total_miles': 138727.0,
  'weeks': 52},
 {'driverId': '21',
  'name': 'Jeff Markham',
  'total_hours': 2751.0,
  'total_miles': 138719.0,
  'weeks': 52},
 {'driverId': '41',
  'name': 'Greg Phillips',
  'total_hours': 2723.0